# EPIC Steane -> autqec -> memory gadget (minimal demo)

This notebook shows, in three short steps, that starting from the EPIC Steane code we can:

1. Find a valid automorphism with `autqec`.
2. Feed that automorphism into the `AutQec` gadget in a memory-style experiment.
3. Verify consistency checks indicating distance is preserved.

In [33]:
import numpy as np

from autqec.graph_auts import valid_clifford_auts_B_rows
from autqec.automorphisms import circ_from_aut, logical_circ_and_pauli_correct

from epic.modules.stabilizers_codes.css_code import CSSCode
from epic.modules.qec_gadgets.transversal_gates.autqec_transversal import AutQecTransversal
from epic.modules.qec_gadgets.logical_resets.init_code import InitCode
from epic.modules.qec_gadgets.readout_code import ReadoutCode

from epic.core.compilation.qec_compiler import QECCompiler
from epic.core.data_structure.pauli import PauliEigenState
from epic.core.language.qec_gadget import AllocCode, FreeCode

print("Imports OK")

Imports OK


## 1) Start from EPIC Steane and find an autqec automorphism

In [34]:
# Steane [[7,1,3]] in EPIC
H = np.array(
    [[0, 0, 0, 1, 1, 1, 1],
     [0, 1, 1, 0, 0, 1, 1],
     [1, 0, 1, 0, 1, 0, 1]],
    dtype=np.uint8,
 )
lX = lZ = [1, 1, 1, 0, 0, 0, 0]
selected_code = CSSCode.from_css_pcm(
    code_name="steaneCode",
    hx=H,
    hz=H,
    logical_qubits=[(lX, lZ)],
)

pcm_symp, var_nodes, check_nodes = selected_code.tanner_graph.parity_check_matrix
H_symp = pcm_symp.toarray().astype(np.uint8)
n = H_symp.shape[1] // 2
H_3bit = np.hstack([H_symp, (H_symp[:, :n] + H_symp[:, n:]) % 2])

auts = valid_clifford_auts_B_rows(H_3bit, bits_3=True, return_order=False)
selected_aut_id = 0
aut_raw = auts[selected_aut_id]

print(f"Code: {selected_code.name}, n={selected_code.n}, k={selected_code.k}, d={selected_code.d}")
print(f"Automorphisms found: {len(auts)}")
print(f"Selected automorphism id: {selected_aut_id}")
print(f"Aut label: {aut_raw}")

Code: steaneCode, n=7, k=1, d=3
Automorphisms found: 3
Selected automorphism id: 0
Aut label: [[7, 19], [8, 20], [9, 21], [13, 16], [14, 17], [15, 18]]


## 2) Convert to AutQec gadget fields and run a memory-style compiled program

In [35]:
def derive_detector_check_map_idx(H_symp_local, S):
    S = np.asarray(S, dtype=np.uint8) % 2
    H_trans = (H_symp_local @ S) % 2
    target = {tuple(int(v) for v in row.tolist()): i for i, row in enumerate(H_symp_local)}
    mapping = {}
    used = set()
    for i, row in enumerate(H_trans):
        sig = tuple(int(v) for v in row.tolist())
        if sig not in target:
            raise ValueError("Could not match transformed stabilizer row.")
        j = target[sig]
        if j in used:
            raise ValueError("Not a permutation of stabilizer rows.")
        mapping[i] = [j]
        used.add(j)
    return mapping

def autqec_circuit_to_gadget_fields(phys_circ, var_nodes_local, check_nodes_local, det_map_idx):
    single_qubit_gates = []
    swaps = []
    two_qubit = {"CNOT", "CX", "CZ", "ISWAP"}

    for op in phys_circ:
        if not isinstance(op, (list, tuple)) or len(op) < 2:
            continue
        gate = str(op[0]).replace("SQRT-X", "SQRT_X")
        raw = op[1] if len(op) == 2 else tuple(op[1:])

        if isinstance(raw, np.ndarray):
            raw = raw.flatten().tolist()
        if isinstance(raw, (list, tuple)):
            flat = []
            for t in raw:
                if isinstance(t, (list, tuple, np.ndarray)):
                    flat.extend(int(x) for x in np.asarray(t).reshape(-1))
                else:
                    flat.append(int(t))
        else:
            flat = [int(raw)]

        # autqec uses 1-based indices
        if gate == "SWAP" and len(flat) >= 2:
            swaps.append((var_nodes_local[flat[0] - 1], var_nodes_local[flat[1] - 1]))
        elif gate in two_qubit:
            for i in range(0, len(flat) - 1, 2):
                single_qubit_gates.append((gate, var_nodes_local[flat[i] - 1]))
                single_qubit_gates.append((gate, var_nodes_local[flat[i + 1] - 1]))
        else:
            for idx in flat:
                single_qubit_gates.append((gate, var_nodes_local[idx - 1]))

    detector_check_map = {
        check_nodes_local[int(cur)]: tuple(check_nodes_local[int(prev)] for prev in prevs)
        for cur, prevs in det_map_idx.items()
    }
    return single_qubit_gates, swaps, detector_check_map

phys_act = circ_from_aut(H_symp, aut_raw)
phys_circ_raw, symp_mat = phys_act.circ()
det_map_idx = derive_detector_check_map_idx(H_symp, symp_mat)
sqg, swps, det_map = autqec_circuit_to_gadget_fields(
    phys_circ_raw, var_nodes, check_nodes, det_map_idx
)

# Mention properties of the selected automorphism
logical_act = logical_circ_and_pauli_correct(H_symp, phys_circ_raw)
logical_circ, _ = logical_act.run()
U_log = np.asarray(logical_act.U_logical_act(), dtype=np.uint8) % 2
num_sq = len(sqg)
num_swaps = len(swps)
detector_map_is_perm = (
    len(det_map) == len(check_nodes)
    and set(det_map.keys()) == set(check_nodes)
    and len([p for v in det_map.values() for p in v]) == len(check_nodes)
)
print("Selected automorphism properties:")
print(f"  aut label: {aut_raw}")
print(f"  logical action matrix shape: {U_log.shape}")
print(f"  logical action matrix:\n{U_log}")
print(f"  logical circuit: {logical_circ}")
print(f"  physical gate entries: {len(phys_circ_raw)}")
print(f"  mapped single-qubit gate applications: {num_sq}")
print(f"  swap count: {num_swaps}")
print(f"  detector-check map is full permutation: {detector_map_is_perm}")

program = [
    AllocCode(target_code=selected_code, code_varname="code", logical_qubits_varnames=["lq_0"]),
    InitCode(targets=["code"], initial_state=PauliEigenState.Z_plus),
    AutQecTransversal(
        targets=["code"],
        single_qubit_gates=sqg,
        swaps=swps,
        detector_check_map=det_map,
        tag="autqec_round",
    ),
    ReadoutCode(targets=["code"], tag="readout"),
    FreeCode(code_varname="code"),
]

compiler_config = {
    "objective_distance": selected_code.d,
    "primitives": {
        "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.apply_gates.simple_gate_application.SimpleGateApplication",
        "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.readouts.naive_readout.NaiveReadout",
        "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.syndrome_extraction.zxcoloring_extraction.ZXColoringExtraction",
    },
}

compiled = QECCompiler(config=compiler_config).compile(program)

obs_tags = [obs.tag for obs in compiled.observables]
selected_obs_tag = next((t for t in obs_tags if "_LZ_" in t), obs_tags[0])

from epic.core.experiment.noise_model import StimLikeNoiseModel

physical_noise = 1e-3
noise_model = StimLikeNoiseModel.from_stim_like_probabilities(
    after_clifford_depolarization=physical_noise,
    after_reset_flip_probability=physical_noise,
    before_measure_flip_probability=physical_noise,
)

stim_program_text_noiseless = compiled.to_stim_program([[selected_obs_tag]])
stim_program_text = compiled.to_stim_program([[selected_obs_tag]], noise_model=noise_model)

print(f"AutQec gadget inserted. Detectors: {len(compiled.detectors)}, observables: {len(compiled.observables)}")
print(f"Noise model added with p={physical_noise}")
print("Generated noisy Stim program (before stim.Circuit conversion):")
print(stim_program_text)

Selected automorphism properties:
  aut label: [[7, 19], [8, 20], [9, 21], [13, 16], [14, 17], [15, 18]]
  logical action matrix shape: (2, 2)
  logical action matrix:
[[1 0]
 [0 1]]
  logical circuit: []
  physical gate entries: 2
  mapped single-qubit gate applications: 0
  swap count: 2
  detector-check map is full permutation: True
AutQec gadget inserted. Detectors: 36, observables: 1
Noise model added with p=0.001
Generated noisy Stim program (before stim.Circuit conversion):
RZ 4 0 1 5 2 6 3
X_ERROR(0.001) 4 0 1 5 2 6 3
RZ 11 9 12 10 7 8
X_ERROR(0.001) 11 9 12 10 7 8
REPEAT 3 {
    H 11 9 10
DEPOLARIZE1(0.001) 11 9 10
    TICK
    CX 10 6 11 2 9 4
DEPOLARIZE2(0.001) 10 6 11 2 9 4
    TICK
    CX 11 5 10 1 9 2
DEPOLARIZE2(0.001) 11 5 10 1 9 2
    TICK
    CX 10 4 9 6 11 0
DEPOLARIZE2(0.001) 10 4 9 6 11 0
    TICK
    CX 10 0 11 6 9 3
DEPOLARIZE2(0.001) 10 0 11 6 9 3
    TICK
    H 11 9 10
DEPOLARIZE1(0.001) 11 9 10
    TICK
    CX 0 7 1 12 6 8
DEPOLARIZE2(0.001) 0 7 1 12 6 8
    T

## 3) Check that the automorphism preserves code structure and logical weight

In [36]:
# Stabilizer-row permutation check
H_trans = (H_symp @ symp_mat) % 2
orig_rows = {tuple(int(v) for v in r.tolist()) for r in H_symp}
trans_rows = {tuple(int(v) for v in r.tolist()) for r in H_trans}
row_perm_ok = orig_rows == trans_rows

# Logical support weight check
idx = {v.id: i for i, v in enumerate(var_nodes)}
orig_w = []
trans_w = []
for lq in selected_code.logical_qubits:
    for lop in (lq.logical_x, lq.logical_z):
        vec = np.zeros(2 * n, dtype=np.uint8)
        for pauli, node in zip(lop.operator.string, lop.target_nodes):
            j = idx[node.id]
            if pauli in ("X", "Y"):
                vec[j] = 1
            if pauli in ("Z", "Y"):
                vec[j + n] = 1
        vt = (vec @ symp_mat) % 2
        w0 = int(np.count_nonzero((vec[:n] | vec[n:]).astype(np.uint8)))
        w1 = int(np.count_nonzero((vt[:n] | vt[n:]).astype(np.uint8)))
        orig_w.append(w0)
        trans_w.append(w1)

logical_weight_ok = min(trans_w) >= min(orig_w)
distance_ok = row_perm_ok and logical_weight_ok

print("Row permutation check:", "PASS" if row_perm_ok else "FAIL")
print("Logical-weight check:", "PASS" if logical_weight_ok else "FAIL")
print("min original logical support weight:", min(orig_w))
print("min transformed logical support weight:", min(trans_w))
print("target distance d:", selected_code.d)
print("DISTANCE-PRESERVATION VERDICT:", "PASS" if distance_ok else "FAIL")

# Stim graphlike-distance check
import stim

circuit = stim.Circuit(stim_program_text)
graphlike_error = circuit.shortest_graphlike_error(ignore_ungraphlike_errors=True)
graphlike_distance = len(graphlike_error)
graphlike_ok = graphlike_distance == selected_code.d

print("Stim shortest_graphlike_error weight:", graphlike_distance)
print("SHORTEST_GRAPHLIKE_ERROR == d:", "PASS" if graphlike_ok else "FAIL")

Row permutation check: PASS
Logical-weight check: PASS
min original logical support weight: 3
min transformed logical support weight: 3
target distance d: 3
DISTANCE-PRESERVATION VERDICT: PASS
Stim shortest_graphlike_error weight: 2
SHORTEST_GRAPHLIKE_ERROR == d: FAIL


### Result

If the last cell prints `PASS`, this demonstrates the full pipeline requested:
- automorphism found with `autqec`,
- injected into the EPIC `AutQec` memory experiment,
- structural and logical-weight checks consistent with distance preservation.

Effective distance is not preserve here, but this is caused by the extraction circuit that is not optimal.